# Stage 2 (ASpanFormer) batching -- correctness + timing smoke test

Standalone, self-contained -- not part of the production pipeline, and does not depend on
`main.ipynb` having been run first. Validates `aspan_batching.py`'s batched forward pass
before trusting `geometry_filter.main([..., '--batch-size', 'N', ...])` for any real run.

**Batching only ever groups candidates whose source/target images resolve to the exact same
`(h,w)` after `resize_with_scale` + `resize_df`** (shape-bucketing, see `aspan_batching.py`'s
module docstring) -- padding *different*-shaped images together was found to corrupt
keypoints near the padded boundary (a mask-unaware bilinear upsample inside the vendored
matcher's flow prediction), so that path is never exercised at all rather than patched.

This notebook mounts Drive itself, copies only the handful of small files it needs (three
scripts + one weights checkpoint -- never the multi-GB source/target corpus archives), and
samples real candidate pairs from the existing `dino_retrieval_manifest.jsonl` already on
Drive at `ImageSimilarity/pipeline_output/retrieval_manifest.jsonl` (the same file the
LightGlue/RoMa ablation notebooks use). It resolves each pair's `source_id`/`target_id` by
reading the corpus zips' file listings directly (no bulk extraction) and extracts only the
handful of images actually needed.

Four things checked, in order:
1. **Bucket statistics** -- across a large real sample, how many distinct shapes actually
   occur, and what fraction of candidates share a bucket with at least one other (i.e. have
   real batching opportunity)?
2. **Correctness A** -- does `--batch-size 1` reproduce today's per-pair path exactly?
3. **Correctness B** -- does batching a real same-shape bucket together match running each
   pair alone?
4. **Timing/VRAM** -- ms/pair and peak memory across a few batch sizes on a real bucket, to
   pick a default.

Do not use `--batch-size > 1` for any run whose output feeds a reported result until 2 and 3
both print PASS on your actual GPU tier.


---
## Section A -- Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import subprocess, sys
from pathlib import Path


def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)


DRIVE_ROOT = Path('/content/drive/MyDrive/ImageSimilarity')
ASPAN_REQS = DRIVE_ROOT / 'ml-aspanformer/requirements.txt'
if ASPAN_REQS.exists():
    _pip('-r', str(ASPAN_REQS))
else:
    print('WARNING: ASpanFormer requirements.txt not found at', ASPAN_REQS)
print('Installs done.')


---
## Section B -- Configuration
Fully self-contained -- reads directly from Drive, no dependency on `main.ipynb` having staged anything.

In [ ]:
from pathlib import Path
import shutil

DRIVE_ROOT  = Path('/content/drive/MyDrive/ImageSimilarity')
LOCAL_ROOT  = Path('/content/stage2_batching_smoketest')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

ASPAN_ROOT           = DRIVE_ROOT / 'ml-aspanformer'
ASPAN_CONFIG         = ASPAN_ROOT / 'configs/aspan/outdoor/aspan_test.py'
ASPAN_WEIGHTS_DRIVE  = DRIVE_ROOT / 'aspanweights/outdoor.ckpt'
ASPAN_WEIGHTS_LOCAL  = LOCAL_ROOT / 'weights/outdoor.ckpt'

# Same file the LightGlue/RoMa ablation notebooks use (ablation/stage2_ablation_colab.ipynb) --
# already uploaded to Drive at this path, contains source_id/target_id (not real paths) for
# ~427k real production candidate pairs.
RETRIEVAL_MANIFEST_DRIVE = DRIVE_ROOT / 'pipeline_output/retrieval_manifest.jsonl'

# Corpus archives -- read directly via zipfile below, NEVER bulk-extracted (they're massive;
# only the handful of images actually sampled get pulled out).
SOURCE_ZIP = DRIVE_ROOT / 'workingsourcecrops.zip'
TARGET_ZIP = DRIVE_ROOT / 'working_targetcrops.zip'

GEOFILTER_SCRIPT_DRIVE = DRIVE_ROOT / 'geometry_filter.py'
ASPAN_BATCHING_DRIVE   = DRIVE_ROOT / 'aspan_batching.py'
# retrieve.py itself isn't run here -- only imported for its IMAGE_EXTENSIONS constant and
# stem convention (retrieve.py:31-33,69-75,191-192), so ID resolution can't drift out of sync
# with how the production pipeline actually assigns source_id/target_id.
RETRIEVE_SCRIPT_DRIVE = DRIVE_ROOT / 'retrieve.py'

LOCAL_ROOT.joinpath('weights').mkdir(parents=True, exist_ok=True)
for src, dst in [
    (GEOFILTER_SCRIPT_DRIVE, LOCAL_ROOT / GEOFILTER_SCRIPT_DRIVE.name),
    (ASPAN_BATCHING_DRIVE, LOCAL_ROOT / ASPAN_BATCHING_DRIVE.name),
    (RETRIEVE_SCRIPT_DRIVE, LOCAL_ROOT / RETRIEVE_SCRIPT_DRIVE.name),
    (ASPAN_WEIGHTS_DRIVE, ASPAN_WEIGHTS_LOCAL),
]:
    shutil.copy2(src, dst)
    print(f'Copied {src.name} -> {dst}')

import sys
if str(LOCAL_ROOT) not in sys.path:
    sys.path.insert(0, str(LOCAL_ROOT))
print('Ready.')


---
## Section C -- Load matcher and sample real pairs

In [ ]:
import importlib
import torch

import geometry_filter as gf
import aspan_batching
importlib.reload(gf)
importlib.reload(aspan_batching)

LONG_DIM = 1024  # matches geometry_filter.py's --long_dim default

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(device))

matcher = gf.load_aspanformer(str(ASPAN_ROOT), str(ASPAN_CONFIG), str(ASPAN_WEIGHTS_LOCAL), device)
print('Matcher loaded.')


In [ ]:
import json
import random
import zipfile

import retrieve  # only for IMAGE_EXTENSIONS + the stem convention, not run


def index_zip_images(zip_path):
    """{stem: zip-internal member name} for every image in the archive, via the
    file listing only -- no extraction. Mirrors retrieve.collect_images()'s own
    extension filter and Path(...).stem convention (retrieve.py:31-33,69-75,191-192),
    since that's exactly how source_id/target_id were assigned in the first place."""
    with zipfile.ZipFile(zip_path) as zf:
        return {
            Path(name).stem: name
            for name in zf.namelist()
            if Path(name).suffix.lower() in retrieve.IMAGE_EXTENSIONS
        }


print('Indexing corpus zips (file listing only, no extraction)...')
source_index = index_zip_images(SOURCE_ZIP)
target_index = index_zip_images(TARGET_ZIP)
print(f'  {len(source_index)} source images, {len(target_index)} target images available.')


def load_and_resolve_pairs(manifest_path, n=24, seed=0):
    rows = []
    with open(manifest_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    random.Random(seed).shuffle(rows)

    resolved, skipped = [], 0
    for row in rows:
        source_member = source_index.get(row.get('source_id'))
        target_member = target_index.get(row.get('target_id'))
        if source_member is None or target_member is None:
            skipped += 1
            continue
        row = dict(row)
        row['_source_member'] = source_member
        row['_target_member'] = target_member
        resolved.append(row)
        if len(resolved) >= n:
            break
    print(f'Resolved {len(resolved)} pairs ({skipped} skipped -- source_id/target_id not found '
          'in the current corpus zips; expected since this manifest predates the current corpus).')
    return resolved


sample_rows = load_and_resolve_pairs(RETRIEVAL_MANIFEST_DRIVE, n=24)

# Extract only the resolved sample's files -- never the whole archive.
source_dir = LOCAL_ROOT / 'sample_images' / 'source'
target_dir = LOCAL_ROOT / 'sample_images' / 'target'
source_dir.mkdir(parents=True, exist_ok=True)
target_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(SOURCE_ZIP) as zf_src, zipfile.ZipFile(TARGET_ZIP) as zf_tgt:
    for row in sample_rows:
        row['source_path'] = zf_src.extract(row.pop('_source_member'), source_dir)
        row['target_path'] = zf_tgt.extract(row.pop('_target_member'), target_dir)

print(f'Extracted {len(sample_rows)} sample pairs to {LOCAL_ROOT / "sample_images"}.')
for r in sample_rows[:3]:
    print(' ', r['source_path'], '<->', r['target_path'])


### Bucket statistics

Local CPU testing found that batching *differently*-shaped images together (via padding)
corrupts keypoints near the padded boundary -- a mask-unaware bilinear upsample inside the
vendored matcher's flow prediction, not a simple bug. `--batch-size > 1` in `geometry_filter.py`
now only ever groups candidates whose source/target images resolve to the *exact same*
`(h,w)` after `resize_with_scale` + `resize_df` -- no padding, no masking, no corruption,
just fewer, larger GPU calls.

Because the long edge is always pinned to exactly `LONG_DIM` and the short edge is floored to
a multiple of 32, there are only `LONG_DIM/32` possible shapes per orientation -- a small,
bounded space real candidates should cluster into, not one where every pair needs its own
singleton batch. This cell checks that directly against your actual corpus, reading shapes
in-memory from the zips (no extraction) so it can look at far more candidates than the small
correctness-check sample above.

In [ ]:
import json
import random
import zipfile
from collections import defaultdict

import cv2
import numpy as np

BUCKET_STATS_SAMPLE = 2000  # shapes are read in-memory (no extraction), so this can be
                             # much larger than the correctness-check sample above.


def shape_from_zip_bytes(zf, member, long_dim, resize_with_scale):
    """Same shape computation as aspan_batching.compute_resized_shape, but decoding
    straight from zip bytes in memory -- for bulk statistics only, never used to
    decide real batch membership (geometry_filter.py always reads real files via
    aspan_batching.compute_resized_shape, so this can't silently drift out of sync
    with what production actually does)."""
    data = zf.read(member)
    gray = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        return None
    resized, _ = resize_with_scale(gray, long_dim)
    h, w = resized.shape[:2]
    df = 32
    return (h // df * df, w // df * df)


def sample_and_bucket(manifest_path, n, seed=1):
    rows = []
    with open(manifest_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    random.Random(seed).shuffle(rows)

    buckets = defaultdict(list)
    shape_cache = {}
    resolved = skipped = 0
    with zipfile.ZipFile(SOURCE_ZIP) as zf_src, zipfile.ZipFile(TARGET_ZIP) as zf_tgt:
        for row in rows:
            if resolved >= n:
                break
            src_member = source_index.get(row.get('source_id'))
            tgt_member = target_index.get(row.get('target_id'))
            if src_member is None or tgt_member is None:
                skipped += 1
                continue
            if src_member not in shape_cache:
                shape_cache[src_member] = shape_from_zip_bytes(zf_src, src_member, LONG_DIM, gf.resize_with_scale)
            if tgt_member not in shape_cache:
                shape_cache[tgt_member] = shape_from_zip_bytes(zf_tgt, tgt_member, LONG_DIM, gf.resize_with_scale)
            src_shape, tgt_shape = shape_cache[src_member], shape_cache[tgt_member]
            if src_shape is None or tgt_shape is None:
                skipped += 1
                continue
            row = dict(row)
            row['_source_member'] = src_member
            row['_target_member'] = tgt_member
            buckets[(src_shape, tgt_shape)].append(row)
            resolved += 1
    print(f'Resolved {resolved} candidates ({skipped} skipped -- IDs not found in the '
          f'current corpus) into {len(buckets)} shape buckets.')
    return buckets


bucket_map = sample_and_bucket(RETRIEVAL_MANIFEST_DRIVE, BUCKET_STATS_SAMPLE)

sizes = sorted((len(v) for v in bucket_map.values()), reverse=True)
total = sum(sizes)
batchable = sum(s for s in sizes if s >= 2)
print(f'Bucket sizes (top 10 of {len(sizes)}): {sizes[:10]}')
if total:
    print(f'{batchable}/{total} candidates ({100 * batchable / total:.1f}%) share a bucket '
          f'with at least one other candidate -- real batching opportunity.')
    print(f'Mean bucket size: {total / len(sizes):.1f}, median: {sizes[len(sizes) // 2]}')


In [ ]:
## Extract the largest real bucket's images -- this is what Sections D/E test below,
## instead of deliberately-mismatched shapes, since that's the only thing production
## code will ever actually batch together.
largest_key = max(bucket_map, key=lambda k: len(bucket_map[k]))
large_bucket_rows = bucket_map[largest_key]
print(f'Largest bucket: shape {largest_key}, {len(large_bucket_rows)} candidates in this sample.')

BUCKET_SAMPLE_N = min(32, len(large_bucket_rows))  # cap for a reasonably fast smoke test
large_bucket_rows = large_bucket_rows[:BUCKET_SAMPLE_N]

bucket_source_dir = LOCAL_ROOT / 'bucket_images' / 'source'
bucket_target_dir = LOCAL_ROOT / 'bucket_images' / 'target'
bucket_source_dir.mkdir(parents=True, exist_ok=True)
bucket_target_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(SOURCE_ZIP) as zf_src, zipfile.ZipFile(TARGET_ZIP) as zf_tgt:
    for row in large_bucket_rows:
        row['source_path'] = zf_src.extract(row.pop('_source_member'), bucket_source_dir)
        row['target_path'] = zf_tgt.extract(row.pop('_target_member'), bucket_target_dir)

print(f'Extracted {len(large_bucket_rows)} real same-shape pairs from the largest bucket.')
for r in large_bucket_rows[:3]:
    print(' ', r['source_path'], '<->', r['target_path'])


---
## Section D -- Correctness

Compares full match dicts on the fields that actually drive downstream decisions
(keypoint counts, raw keypoint coordinates) -- not just a summary statistic, so a
subtle per-point drift can't hide behind a matching count.

In [ ]:
import numpy as np


def compare_match(a, b, atol=1e-4):
    """Compare two run_aspan_pair-shaped dicts. Returns (ok, list_of_issue_strings)."""
    issues = []
    for key in ['raw_keypoint_count', 'filtered_keypoint_count']:
        if a[key] != b[key]:
            issues.append(f'{key}: {a[key]} != {b[key]}')
    for key in ['raw_mkpts0_resized', 'raw_mkpts1_resized']:
        av, bv = a[key], b[key]
        if av.shape != bv.shape:
            issues.append(f'{key}: shape {av.shape} != {bv.shape}')
        elif av.size and not np.allclose(av, bv, atol=atol):
            max_diff = float(np.max(np.abs(av - bv)))
            issues.append(f'{key}: max abs diff {max_diff:.6g} (atol={atol})')
    return (len(issues) == 0, issues)


In [ ]:
print("=== Correctness A: --batch-size 1 vs today's per-pair path ===")
all_ok = True
for i, row in enumerate(sample_rows[:10]):
    src, tgt = row['source_path'], row['target_path']
    today = gf.run_aspan_pair(matcher, src, tgt, LONG_DIM, device)
    batched = aspan_batching.run_aspan_batch(
        matcher, [row], LONG_DIM, device, gf.resize_with_scale, gf.keypoints_to_original, gf.run_ransac
    )[0]
    ok, issues = compare_match(today, batched)
    all_ok &= ok
    print(f'[{i}] {"OK" if ok else "MISMATCH"}  raw_kpts={today["raw_keypoint_count"]}')
    for issue in issues:
        print('      ', issue)

print()
if all_ok:
    print('PASS: --batch-size 1 reproduces the current per-pair path.')
else:
    print('FAIL: see mismatches above -- do not trust any --batch-size > 1 run until this passes.')


In [ ]:
print('=== Correctness B: real bucketed batch (same-shape group) vs solo ===')
print(f'Using the largest real bucket found above: shape {largest_key}, {len(large_bucket_rows)} pairs.')
print('(Batching different-shaped pairs together is no longer something production code '
      'does at all -- geometry_filter.py only ever batches within a shape bucket -- so this '
      'is now the actual scenario to validate, not a worst-case stress test.)')

TEST_BATCH_SIZE = 8  # mirrors a real --batch-size value. large_bucket_rows is chunked into
                      # groups of this size for the "batched" side below -- exactly how
                      # geometry_filter.py's real batched path chunks within a bucket
                      # (`for i in range(0, len(bucket_rows), args.batch_size): flush(...)`),
                      # not one giant forward pass over the whole sample. Lower this if you
                      # hit OOM below.

if device.type == 'cuda':
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info(device)
    print(f'GPU memory free: {free_b / 1e9:.1f} GB / {total_b / 1e9:.1f} GB total')

print(f'\nRunning {len(large_bucket_rows)} solo (batch=1) calls...')
solo_results = []
for i, row in enumerate(large_bucket_rows):
    solo_results.append(
        aspan_batching.run_aspan_batch(
            matcher, [row], LONG_DIM, device, gf.resize_with_scale, gf.keypoints_to_original, gf.run_ransac
        )[0]
    )
    print(f'  solo {i + 1}/{len(large_bucket_rows)}')
print('Solo calls done.')

if device.type == 'cuda':
    torch.cuda.empty_cache()  # the sequential solo calls above can leave the allocator
                               # fragmented even though each one succeeded -- clear before
                               # the batched attempt, which needs one large contiguous chunk

num_chunks = (len(large_bucket_rows) + TEST_BATCH_SIZE - 1) // TEST_BATCH_SIZE
print(f'\nRunning {num_chunks} batched chunk(s) of up to {TEST_BATCH_SIZE} pairs each...')
batched_results = []
oom = False
try:
    for chunk_idx, i in enumerate(range(0, len(large_bucket_rows), TEST_BATCH_SIZE), start=1):
        chunk = large_bucket_rows[i : i + TEST_BATCH_SIZE]
        print(f'  batched chunk {chunk_idx}/{num_chunks} (size {len(chunk)})')
        batched_results.extend(
            aspan_batching.run_aspan_batch(
                matcher, chunk, LONG_DIM, device, gf.resize_with_scale, gf.keypoints_to_original, gf.run_ransac
            )
        )
    print('Batched calls done.')
except RuntimeError as exc:
    if 'out of memory' not in str(exc).lower():
        raise
    oom = True
    print(f'\nOOM at TEST_BATCH_SIZE={TEST_BATCH_SIZE} on this GPU.')
    print('Lower TEST_BATCH_SIZE above (or BUCKET_SAMPLE_N a few cells up) and re-run this cell.')
    print(f'  ({exc})')

if not oom:
    print()
    all_ok = True
    for i, (solo, batched) in enumerate(zip(solo_results, batched_results)):
        ok, issues = compare_match(solo, batched)
        all_ok &= ok
        print(f'[{i}] {"OK" if ok else "MISMATCH"}  raw_kpts solo={solo["raw_keypoint_count"]} batched={batched["raw_keypoint_count"]}')
        for issue in issues:
            print('      ', issue)

    print()
    if all_ok:
        print("PASS: bucketed batching of real same-shape pairs is exact.")
    else:
        print('FAIL: unexpected -- bucketing should guarantee mask=None for every pair here. '
              'Check that every row in large_bucket_rows truly resolved to the same shape.')


---
## Section E -- Timing / VRAM sweep

Only meaningful once both correctness checks above print PASS. Uses `large_bucket_rows`
(the largest real same-shape bucket found above) and reports ms/pair and peak VRAM per
batch size, so you can pick a default `--batch-size` for your actual Colab GPU tier. Real
batches in production are always within-bucket, so timing a real bucket is the representative
number, not the small correctness-check sample.

In [ ]:
import time


def benchmark_batch_size(rows, batch_size, device):
    if device.type == 'cuda':
        torch.cuda.empty_cache()  # start each batch size with a clean allocator state,
                                   # not carrying fragmentation over from the previous size
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats(device)
    num_chunks = (len(rows) + batch_size - 1) // batch_size
    print(f'  timing batch_size={batch_size} ({num_chunks} chunk(s))...')
    start = time.perf_counter()
    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i + batch_size]
        aspan_batching.run_aspan_batch(
            matcher, chunk, LONG_DIM, device, gf.resize_with_scale, gf.keypoints_to_original, gf.run_ransac
        )
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    peak_mb = (torch.cuda.max_memory_allocated(device) / 1e6) if device.type == 'cuda' else float('nan')
    return elapsed / len(rows), peak_mb


BATCH_SIZES = [1, 2, 4, 8, 16]
# Increase BUCKET_SAMPLE_N a few cells up (and re-run from there) for a steadier
# timing estimate or to test larger batch sizes, if the largest bucket has enough rows.

if device.type == 'cuda':
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info(device)
    print(f'GPU memory free: {free_b / 1e9:.1f} GB / {total_b / 1e9:.1f} GB total\n')

print(f'{"batch_size":>10} | {"ms/pair":>10} | {"peak VRAM (MB)":>15}')
print('-' * 42)
for bs in BATCH_SIZES:
    if bs > len(large_bucket_rows):
        print(f'{bs:>10} | skipped (need >= {bs} same-shape pairs, have {len(large_bucket_rows)})')
        continue
    try:
        ms_per_pair, peak_mb = benchmark_batch_size(large_bucket_rows, bs, device)
        print(f'{bs:>10} | {ms_per_pair * 1000:>10.1f} | {peak_mb:>15.1f}')
    except RuntimeError as exc:
        if 'out of memory' not in str(exc).lower():
            raise
        print(f'{bs:>10} | OOM -- larger batch sizes will OOM too, stopping here')
        break

print()
print("Pick the largest batch size with VRAM headroom and the best ms/pair -- expect "
      "diminishing returns once GPU compute saturates (see the speedup discussion earlier "
      "in this session for why).")
